# Solutions Guide: Data Cleaning and Feature Engineering
Course/Module: Machine Learning & Data Preprocessing
Topic: Applied Missing Value Handling and Categorical Encoding
Language: Python (using Pandas, NumPy, and Seaborn/Matplotlib)

## Part 1: Handling Missing Values
### 1. Data Loading and Missing Value Audit

In [ ]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('customer_data.csv')

# Calculate missing values and percentages
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Combine into a DataFrame for readability
missing_audit = pd.DataFrame({
'Missing_Count': missing_counts,
'Percentage': missing_percentages
})

# Identify top 3 columns with highest missing rates
top_3_missing = missing_audit.sort_values(by='Percentage', ascending=False).head(3)
print(top_3_missing)

### 2. Threshold-Based Column Dropping

In [ ]:
# Identify columns with more than 40% missing data
threshold = 40.0
cols_to_drop = missing_audit[missing_audit['Percentage'] > threshold].index

# Drop the columns
df = df.drop(columns=cols_to_drop)

# Justification:
# Dropping these columns is preferred because when nearly half the data is missing,
# any imputation strategy (mean, median, mode) will introduce massive artificial bias
# and destroy the original variance/distribution of the feature.

### 3. Row-wise Deletion (Listwise Deletion)

In [ ]:
print(f"Original shape: {df.shape}")

# Identify columns with > 0% and < 5% missing values
low_missing_cols = missing_audit[(missing_audit['Percentage'] > 0) &
(missing_audit['Percentage'] < 5.0)].index

# Drop rows containing missing values in ONLY these specific columns
df = df.dropna(subset=low_missing_cols)

print(f"New shape: {df.shape}")

### 4. Skewness and Distribution-Based Imputation

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plotting distributions (Optional for the script, required by prompt)
sns.histplot(df['Customer_Age'], kde=True)
plt.show()

sns.histplot(df['Annual_Income'], kde=True)
plt.show()

# Assuming Customer_Age is normally distributed (use mean)
# and Annual_Income is highly right-skewed (use median)
df['Customer_Age'] = df['Customer_Age'].fillna(df['Customer_Age'].mean())
df['Annual_Income'] = df['Annual_Income'].fillna(df['Annual_Income'].median())

### 5. Group-Based Imputation (Hard Basic)

In [ ]:
# Impute missing Credit_Score based on the mean of their specific Employment_Status
df['Credit_Score'] = df.groupby('Employment_Status')['Credit_Score'].transform(
lambda x: x.fillna(x.mean())
)

### 6. Categorical Mode Imputation

In [ ]:
# Find the mode (most frequent category)
mode_category = df['Preferred_Shopping_Category'].mode()[0]

# Fill missing values
df['Preferred_Shopping_Category'] = df['Preferred_Shopping_Category'].fillna(mode_category)

### 7. "Unknown" Category Assignment

In [ ]:
# Fill missing feedback with a distinct "No Feedback" category
df['Customer_Feedback'] = df['Customer_Feedback'].fillna("No Feedback")

### 8. Sequential Imputation (Forward Fill)

In [ ]:
# Sort by date first to ensure logical chronological order
df = df.sort_values(by='Account_Creation_Date')

# Apply forward fill
df['Last_Login_Date'] = df['Last_Login_Date'].fillna(method='ffill')

# Logical Flaw Scenario:
# Forward fill assumes the missing value should be exactly the same as the previous row.
# If the dataset is not grouped by user first, you might accidentally forward-fill
# User B's missing login date with User A's login date, which is completely entirely invalid.

### 9. The Missing Indicator Creation

In [ ]:
# Create the boolean indicator column (1 if missing, 0 otherwise)
df['Premium_Status_Missing'] = df['Has_Premium_Subscription'].isnull().astype(int)

# Impute the original missing values with 0
df['Has_Premium_Subscription'] = df['Has_Premium_Subscription'].fillna(0)

### 10. Final Sanity Check

In [ ]:
# Assert statement to ensure no missing values remain
assert df.isnull().sum().sum() == 0, "Error: There are still missing values in the DataFrame!"
print("Sanity check passed: No missing values remain.")

## Part 2: Handling Categorical Data
### 11. String Standardization Before Encoding

In [ ]:
# Strip whitespace and convert to lowercase
df['Subscription_Tier'] = df['Subscription_Tier'].str.strip().str.lower()

### 12. Binary Mapping

In [ ]:
# Map text to binary integers
binary_map = {'Yes': 1, 'No': 0, 'yes': 1, 'no': 0}
df['Is_Active'] = df['Is_Active'].map(binary_map)

### 13. Custom Ordinal Encoding

In [ ]:
# Define the order mapping
edu_map = {
'High School': 1,
'Bachelors': 2,
'Masters': 3,
'PhD': 4
}

# Apply the mapping
df['Education_Level'] = df['Education_Level'].map(edu_map)

### 14. Nominal Encoding (One-Hot)

In [ ]:
# Standard One-Hot Encoding
df = pd.get_dummies(df, columns=['Payment_Method'])

### 15. The Dummy Variable Trap

In [ ]:
# One-Hot Encoding dropping the first column
df = pd.get_dummies(df, columns=['Payment_Method'], drop_first=True)

# Explanation:
# Dropping the first category prevents perfectly collinear variables (multicollinearity).
# If you have 3 categories (A, B, C) and you know a row is NOT A and NOT B,
# it must be C. Therefore, the 3rd column is redundant and breaks the mathematical
# assumptions of linear regression (specifically, matrix inversion).

### 16. Grouping Rare Categories (Hard Basic)

In [ ]:
# Get value counts of job titles
job_counts = df['Job_Title'].value_counts()

# Identify titles that appear less than 5 times
rare_jobs = job_counts[job_counts < 5].index

# Replace rare titles with 'Other_Jobs'
df['Job_Title'] = df['Job_Title'].replace(rare_jobs, 'Other_Jobs')

### 17. Frequency / Count Encoding

In [ ]:
# Calculate frequencies
city_frequencies = df['City_of_Residence'].value_counts()

# Map the frequencies back to the dataframe
df['City_of_Residence'] = df['City_of_Residence'].map(city_frequencies)

### 18. Target Encoding (Mean Encoding)

In [ ]:
# Calculate mean Total_Spent for each Region
region_target_means = df.groupby('Region')['Total_Spent'].mean()

# Replace Region text with the calculated target mean
df['Region'] = df['Region'].map(region_target_means)

### 19. Splitting and Encoding Composite Strings

In [ ]:
# Split by hyphen and grab the first element (index 0)
df['Product_Category'] = df['Product_Code'].str.split('-').str[0]

# One-hot encode the newly extracted category
df = pd.get_dummies(df, columns=['Product_Category'], drop_first=True)

# (Optional) Drop the original composite column
df = df.drop(columns=['Product_Code'])

### 20. Final Dataset Compilation

In [ ]:
# Drop any remaining object (text/string) columns
df_final = df.select_dtypes(exclude=['object'])

# Display datatypes to ensure they are all int or float
print(df_final.dtypes)

# Display the first 5 rows of the model-ready dataset
print(df_final.head())